In [1]:
import os

try:
    os.makedirs("src")
except FileExistsError:
    print(f"directory already exists!")

directory already exists!


# loading the raw data from `yfinance`

this serves as the base function for loading all data

In [2]:
%%writefile src/load_data.py

import pandas as pd
import yfinance as yf
import polars as pl

def load_stocks(stocks: list, start: str, end: str, use_polars: bool = True):
    if len(stocks) > 1:
        raise ValueError("can only do one stock forecast at a time")
    df = yf.download(stocks, start, end)
    df.index = pd.to_datetime(df.index)
    df.columns = (
        pd.MultiIndex.from_tuples(df.columns) 
        if not isinstance(df.columns, pd.MultiIndex) else df.columns
    )
    df.columns = df.columns.set_names(["Field", "Ticker"])
    df.index.name = "Date"
    df = df.apply(pd.to_numeric, errors="coerce")

    df_out = (
        df.swaplevel("Field", "Ticker", axis=1)
        .sort_index(axis=1)
        .stack("Ticker", future_stack=True)
        .reset_index()
    )

    df_out = df_out.rename(columns=str.lower)

    return pl.from_pandas(df_out) if use_polars else df_out

Overwriting src/load_data.py


# transform the raw data by adding shift columns

so far, most useful for `xgboost` modeling  
not so useful for `sarimax` (minus time variables)  
we'll see for others...

In [3]:
%%writefile src/data_etl.py

import polars as pl
from src.load_data import load_stocks

def prep_columns(df: pl.DataFrame, col: str) -> pl.DataFrame:
    if col == "move":
        df = df.with_columns(
            (pl.col("close") - pl.col("open")).alias(col)
        )

    df_out =  (
        df.select(
            [
                "date",
                "ticker",
                "volume",
                col
            ]
        )
        .sort([pl.col("ticker"), pl.col("date")], descending=False)
        .with_columns(
            pl.col(col).shift(1).over("ticker").alias(f"prev1_{col}"),
            pl.col(col).shift(7).over("ticker").alias(f"prev7_{col}"),
            pl.col(col).shift(30).over("ticker").alias(f"prev30_{col}"),
        )
        .with_columns(
            pl.col(col)
            .rolling_mean(window_size=7, min_samples=2)
            .shift(1)
            .over("ticker")
            .alias(f"{col}_rolling_mean_7")
        )
        .with_columns(
            pl.col(col)
            .rolling_std(window_size=7, min_samples=2)
            .shift(1)
            .over("ticker")
            .alias(f"{col}_rolling_std_7")
        )
    )

    return df_out


def prep_data_frame(df):
    markers = ["open", "close", "move"]
    df_out = None

    for marker in markers:
        df_prep = prep_columns(df, marker)
        if df_out is None:
            df_out = df_prep
        else:
            df_out = df_prep.join(
                df_out, on=["date", "ticker", "volume"], how="inner"
            )
    
    return (
        df_out.with_columns(
            pl.col("date").dt.weekday().alias("dow")
        )
        .with_columns(
            pl.col("date").dt.month().alias("month")
        )
        .with_columns(
            pl.when(pl.col("dow").is_in([0, 4]))
            .then(pl.lit(1))
            .otherwise(pl.lit(0))
            .alias("mon_or_fri")
        )
    )

def build_dataset(df: pl.DataFrame, label: str = "close") -> pl.DataFrame:
    df_feat = prep_data_frame(df)

    if label == "close":
        df_feat = df_feat.with_columns(pl.col("close").shift(-1).alias("label"))
    elif label == "move":
        df_feat = df_feat.with_columns(pl.col("move").shift(-1).alias("label"))
    else:
        raise ValueError("label must be one of ['close', 'move']")
    
    return df_feat.drop_nulls()

## might not get to do this as the data is empty past 1/30/18
def build_df_with_indices(
    df: pl.DataFrame, label: str, start: str, end: str
) -> pl.DataFrame:
    indices_list = ["SPY", "QQQ", "IWM", "VXX", "UUP", "HYG", "LQD"]
    df_idx_out = None

    for idx in indices_list:
        idx_cl = idx.replace("^", "")

        df_idx = load_stocks([idx], start, end).select(
            pl.col("date"),
            pl.col("close").alias(f"{idx_cl}_close"),
            pl.col("volume").alias(f"{idx_cl}_volume")
        )

        if df_idx_out is None:
            df_idx_out = df_idx
        else:
            df_idx_out = df_idx_out.join(df_idx, on=["date"], how="inner")
    
    df_ticker = build_dataset(df, label)

    df_out = df_ticker.join(df_idx_out, on=["date"], how="inner")
    
    return df_out

Overwriting src/data_etl.py


In [4]:
# import polars as pl
# from src.load_data import *
# load_stocks(["SPY"], "2010-10-01", "2025-11-19").sort(pl.col("date"), descending=True)

# prep the data for modeling

mostly just used for train-test splits but different models call for different  
data parsing (even if only slightly)

In [5]:
%%writefile src/model_preprocess.py

from datetime import datetime
import polars as pl
import pandas as pd
from typing import Tuple

def train_test_split_cutoff(
    df: pl.DataFrame, cutoff: datetime, label: str
) -> Tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    train = df.filter(pl.col("date") < cutoff)
    test = df.filter(pl.col("date") >= cutoff)

    X_train, X_test = (
        train.drop(label, "ticker", "date"),
        test.drop(label, "ticker", "date")
    )
    y_train, y_test = train[label], test[label]

    return X_train, X_test, y_train, y_test

def split_ar_on_cutoff(
    df: pl.DataFrame, cutoff: datetime, label: str, exog_feats: list
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train = df.filter(pl.col("date") < cutoff)
    eval = df.filter(pl.col("date") >= cutoff)

    train_pd = train.to_pandas()
    eval_pd = eval.to_pandas()

    chg_cols = [
        # f"prev1_{label}",
        # f"prev7_{label}",
        # f"prev30_{label}",
        # f"{label}_rolling_mean_7",
        f"{label}_rolling_std_7"
    ]

    cols_list = [label] + exog_feats + chg_cols

    train_pd = train_pd[cols_list]
    eval_pd = eval_pd[cols_list]

    return train_pd, eval_pd


Overwriting src/model_preprocess.py


leave this here for now  
trying to mess with `argparse` so it can run in the command line but i'll save  
that for last....

In [6]:
# import argparse

# ### parameters (use argparse module)

# # model hyperparameters
# DEFAULT_NUM_ESTIMATORS = 100
# DEFAULT_LEARNING_RATE = 0.01

# parser = argparse.ArgumentParser(
#     description="model hyperparameters"
# )

# parser.add_argument(
#     "-NUM_ESTIMATORS",
#     type=int,
#     default=DEFAULT_NUM_ESTIMATORS,
#     help="number of estimators"
# )
# parser.add_argument(
#     "-LEARNING_RATE",
#     type=float,
#     default=DEFAULT_LEARNING_RATE,
#     help="how fast the model learns"
# )

# # data parameters
# DEFAULT_LABEL = "move"

# parser.add_argument(
#     "-START_DATE",
#     type=str,
#     default=None,
#     help="data training start date"
# )

# parser.add_argument(
#     "-END_DATE",
#     type=str,
#     default=None,
#     help="data training end date"
# )

# parser.add_argument(
#     "-STOCKS",
#     type=list,
#     default=None,
#     help="stocks to forecast"
# )

# parser.add_argument(
#     "-LABEL",
#     type=str,
#     default=DEFAULT_LABEL,
#     help="one of 'move', 'open', 'close'; which of these values to forecast"
# )

# # cutoff value
# parser.add_argument(
#     "-CUTOFF",
#     type=datetime,
#     default=None,
#     help="cutoff value for train/test split"
# )

# # create args
# args = parser.parse_args()

# NUM_ESTIMATORS = args.NUM_ESTIMATORS
# LEARNING_RATE = args.LEARNING_RATE
# STOCKS = args.STOCKS
# START_DATE = args.START_DATE
# END_DATE = args.END_DATE
# LABEL = args.LABEL
# CUTOFF = args.CUTOFF

# NOW build the model

# training the model

currently only has `xgboost` model  
once `sarimax` is done (done researching/toying with it) i'll add to this module

In [ ]:
%%writefile src/train_model.py

import polars as pl
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from datetime import datetime

from src.load_data import load_stocks
from src.data_etl import *
from src.model_preprocess import train_test_split_cutoff


def train_xgb_model(
    stocks: list,
    start_date: str,
    end_date: str,
    cutoff: datetime,
    label: str,
    n_estimators: int,
    learning_rate: float,
):
    df_raw = load_stocks(
        stocks=stocks,
        start=start_date,
        end=end_date,
        use_polars=True
    )

    df_feat = build_dataset(df=df_raw, label=label)

    X_train, X_test, y_train, y_test = train_test_split_cutoff(
        df=df_feat, cutoff=cutoff, label="label"
    )

    model = xgb.XGBRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror"
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)

    feature_cols = X_train.to_pandas().columns.tolist()

    return model, mse, df_feat, feature_cols

Overwriting src/train_model.py


# forecasting

this predicts future unseen values (not just test data)

In [8]:
%%writefile src/forecaster.py

import polars as pl
import xgboost as xgb
from datetime import timedelta

from src.load_data import load_stocks
from src.data_etl import *


class XGBStockForecaster:
    def __init__(
        self, model: xgb.XGBRegressor, feature_cols: list, label: str
    ):
        self.model = model
        self.feature_cols = feature_cols
        self.label = label
    
    def _predict_from_features(self, df_feat: pl.DataFrame) -> float:
        row_pd = df_feat.select(self.feature_cols).tail(1).to_pandas()
        preds = self.model.predict(row_pd)
        return float(preds[0])
    
    def forecast_horizon(self, df_raw: pl.DataFrame, days: int) -> pl.DataFrame:
        df_current = df_raw.clone()

        forecast_dates = []
        forecast_values = []

        for _ in range(days):
            df_feat = prep_data_frame(df_current)
            pred = self._predict_from_features(df_feat)
            last_date = df_current["date"][-1]
            next_date = last_date + timedelta(days=1)

            while next_date.weekday() >= 5:
                next_date = next_date + timedelta(days=1)
            
            forecast_dates.append(next_date)
            forecast_values.append(pred)

            last_row = df_current.tail(1)

            date_dtype = df_current.schema["date"]

            new_row = last_row.with_columns(
                pl.lit(next_date).cast(date_dtype).alias("date"),
                pl.lit(pred).alias(f"{self.label}")
            )

            df_current = df_current.vstack(new_row)
        
        return pl.DataFrame(
            {
                "date": forecast_dates,
                f"pred_{self.label}": forecast_values
            }
        )

Overwriting src/forecaster.py


# full modeling pipeline

raw data -> ETL -> split -> train a model -> evaluate model training -> forecast

In [9]:
%%writefile src/pipeline.py

import polars as pl
from datetime import datetime
from typing import Tuple

from src.load_data import load_stocks
from src.train_model import train_xgb_model
from src.forecaster import XGBStockForecaster

def train_and_forecast_xgb(
    ticker: str,
    start_date: str,
    end_date: str,
    cutoff: datetime,
    horizon_days: int,
    label: str = "close",
    n_estimators: int = 200,
    learning_rate: float = 0.05
) -> Tuple[pl.DataFrame, float]:
    stocks = [ticker]

    model, mse, df_feat, feature_cols = train_xgb_model(
        stocks=stocks,
        start_date=start_date,
        end_date=end_date,
        cutoff=cutoff,
        label=label,
        n_estimators=n_estimators,
        learning_rate=learning_rate
    )

    df_raw = load_stocks(stocks, start_date, end_date)

    forecaster = XGBStockForecaster(model, feature_cols, label=label)
    forecasts_df = forecaster.forecast_horizon(df_raw, days=horizon_days)

    return forecasts_df, mse

Overwriting src/pipeline.py


In [5]:
%%writefile src/fit_sarimax_model.py

import polars as pl
import pandas as pd
import pmdarima as pm 
from sklearn.metrics import mean_squared_error
from datetime import datetime, timedelta

from src.load_data import load_stocks
from src.data_etl import *
from src.model_preprocess import split_ar_on_cutoff


def fit_sarimax(
    ticker: str,
    start_date: str,
    end_date: str,
    label: str,
    cutoff: datetime,
    days: int,
    eval_mode: bool = True
):
    stock = [ticker]

    df_raw = load_stocks(stock, start_date, end_date)
    df_idx = build_df_with_indices(df_raw, label, start_date, end_date)

    feats = [
        "date",
        "dow",
        "month",
        "mon_or_fri",
        "volume",
        "SPY_close",
        "SPY_volume",
        "QQQ_close",
        "QQQ_volume",
        "IWM_close",
        "IWM_volume",
        "VXX_close",
        "VXX_volume",
        "UUP_close",
        "UUP_volume",
        "HYG_close",
        "HYG_volume",
        "LQD_close",
        "LQD_volume"
    ]

    exog_cols = [feat for feat in feats if feat != "date"]

    _sarima_hyperparams = {
        "start_p": 1,
        "start_q": 1,
        "test": "adf",
        "max_p": 3,
        "max_q": 3,
        "m": 5,
        "start_P": 0,
        "seasonal": True,
        "d": None,
        "D": None,
        "trace": False,
        "error_action": "ignore",
        "suppress_warnings": True,
        "stepwise": True
    }
    
    if eval_mode:
        df_train, df_eval = split_ar_on_cutoff(df_idx, cutoff, "close", feats)

        sarimax_model = pm.auto_arima(
            df_train[[label]],
            exogenous=df_train[exog_cols],
            **_sarima_hyperparams
        )

        fitted, confint = sarimax_model.predict(
            n_periods=len(df_eval),
            return_conf_int=True,
            exogenous=df_eval[exog_cols]
        )

        fitted = pd.DataFrame(fitted, columns=["pred"]).reset_index(drop=True)
        df_eval["pred"] = fitted["pred"]
        mse = mean_squared_error(df_eval[["close"]], df_eval[["pred"]])
        
        return mse
    else:
        df = df_idx.to_pandas()

        sarimax_model = pm.auto_arima(
            df[[label]],
            exogenous=df[exog_cols],
            **_sarima_hyperparams
        )

        fitted, confint = sarimax_model.predict(
            n_periods=days,
            return_conf_int=True,
            exogenous=df[exog_cols]
        )
        fitted = pd.DataFrame(fitted, columns=[f"pred_{label}"]).reset_index(
            drop=True
        )
        ci_series = pd.DataFrame(
            confint, columns=["lower_bound", "upper_bound"]
        )

        forecast_dates = []
        last_date = df["date"].iloc[-1]
        
        for d in range(days):
            next_date = last_date + timedelta(days=d)

            while next_date.weekday() >= 5:
                next_date = next_date + timedelta(days=d)
            
            forecast_dates.append(next_date)

        df_out = pd.DataFrame({"date": forecast_dates})
        
        df_out[f"pred_{label}"] = fitted[f"pred_{label}"]
        df_out["lower_bound"] = ci_series["lower_bound"]
        df_out["upper_bound"] = ci_series["upper_bound"]

        return df_out

def sarimax_wrapper(
    ticker: str,
    start_date: str,
    end_date: str,
    label: str,
    cutoff: datetime,
    days: int
):
    mse = fit_sarimax(
        ticker,
        start_date,
        end_date,
        label,
        cutoff,
        days,
        eval_mode=True
    )

    forecasts = fit_sarimax(
        ticker,
        start_date,
        end_date,
        label,
        cutoff,
        days,
        eval_mode=False
    )

    return pl.from_pandas(forecasts), mse

Overwriting src/fit_sarimax_model.py


In [ ]:
# %%writefile workflows/train_and_forecast_model.py

from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
from timeit import default_timer as timer

import warnings
warnings.filterwarnings("ignore")

from src.pipeline import train_and_forecast_xgb
from src.fit_sarimax_model import *

end_date = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - relativedelta(years=5)).strftime("%Y-%m-%d")
cutoff = (datetime.today() - relativedelta(months=2))

ticker = "AAPL"

timer_xgb_start = timer()

xgb_forecasts, xgb_mse = train_and_forecast_xgb(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    cutoff=cutoff,
    horizon_days=10,
    label="close",
    n_estimators=200,
    learning_rate=0.05
)
timer_xgb_end = timer() - timer_xgb_start

print(f"\nXGBoost test MSE on holdout: {xgb_mse:.4f}\n")
print(f"XGBoost run duration: {timer_xgb_end:.5f} seconds\n\n")
print(xgb_forecasts)
print("\n\n")

timer_smax_start = timer()

smax_forecasts, smax_mse = sarimax_wrapper(
    ticker=ticker,
    start_date=start_date,
    end_date=end_date,
    label="close",
    cutoff=cutoff,
    days=10,    
)
timer_smax_end = timer() - timer_smax_start

print(f"\nSARIMAX test MSE on holdout: {smax_mse:.4f}\n")
print(f"SARIMAX run duration: {timer_smax_end:.5f} seconds\n\n")
print(smax_forecasts)
print("\n\n")

Overwriting workflows/train_and_forecast_model.py
